# LeetCode 3959 — Check Good Integer bằng Python

Link đề: https://leetcode.com/problems/check-good-integer/ · Độ khó: **Easy** · Chủ đề: Math, Simulation

Skeleton LeetCode:

```python
class Solution:
    def checkGoodInteger(self, n: int) -> bool:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: gọi `digitSum` là tổng các chữ số, `squareSum` là tổng bình phương các chữ số; `n` là "tốt" khi `squareSum - digitSum >= 50`.
2. Biết tách chữ số bằng `divmod(n, 10)` hoặc `str(n)`, và tính **hai tổng trong một lượt duyệt**.
3. Nhận ra **bước đột phá**: `squareSum - digitSum = sum(d * (d - 1))`, mỗi chữ số đóng góp độc lập; có thể dựng **bảng 10 ô** và dừng sớm khi đã đạt `50`.
4. Viết 4 cách (hai tổng riêng, một lượt duyệt, bảng đóng góp, dừng sớm) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: dùng `>` thay `>=`, nhầm `(tổng)^2` với tổng bình phương, đảo thứ tự phép trừ, đếm số chữ số thay vì giá trị.

## 2. Đề bài & ví dụ

Cho số nguyên dương `n`. Đặt:

- `digitSum` = tổng các chữ số của `n`;
- `squareSum` = tổng **bình phương** các chữ số của `n`.

`n` được gọi là **số tốt** nếu `squareSum - digitSum >= 50`. Trả về `True` nếu `n` tốt, ngược lại `False`.

| Ví dụ | `n` | Output | Giải thích |
|---|---|---|---|
| 1 | `1000` | `False` | Chữ số `1,0,0,0`: `digitSum = 1`, `squareSum = 1`, hiệu `0 < 50` |
| 2 | `19` | `True` | `digitSum = 10`, `squareSum = 1 + 81 = 82`, hiệu `72 >= 50` |

Constraints: `1 <= n <= 10^9`.

Chú ý: so sánh dùng **lớn hơn hoặc bằng** (`>= 50`): hiệu đúng bằng `50` vẫn là số tốt.

In [1]:
for n in (1000, 19):
    digits = [int(c) for c in str(n)]
    ds, ss = sum(digits), sum(d * d for d in digits)
    print(f"n={n}: chữ số={digits}, digitSum={ds}, squareSum={ss}, hiệu={ss - ds}, tốt? {ss - ds >= 50}")

n=1000: chữ số=[1, 0, 0, 0], digitSum=1, squareSum=1, hiệu=0, tốt? False
n=19: chữ số=[1, 9], digitSum=10, squareSum=82, hiệu=72, tốt? True


## 3. Kiến thức nền

### 3.1. Tách chữ số bằng `divmod(n, 10)`

`divmod(n, 10)` cho `(n // 10, n % 10)`: `n % 10` là chữ số cuối, `n // 10` là phần còn lại. Lặp tới khi `n == 0`.

In [2]:
n = 19
ds = []
while n:
    n, d = divmod(n, 10)
    ds.append(d)
print(ds)

[9, 1]


### 3.2. Chuỗi: `str(n)` và `int(ch)`

`[int(c) for c in str(19)]` cho `[1, 9]`. Ngắn hơn nhưng phải đổi qua lại giữa chuỗi và số.

In [3]:
print([int(c) for c in str(19)], list(map(int, str(1000))))

[1, 9] [1, 0, 0, 0]


### 3.3. `sum` với biểu thức sinh và bình phương `d * d` hay `d ** 2`

`sum(d * d for d in digits)` cộng bình phương mà không tạo list. `d ** 2` cũng được; `d * d` nhanh hơn một chút. Cẩn thận: `(sum(digits)) ** 2` là **bình phương của tổng**, khác **tổng của bình phương**.

In [4]:
digits = [1, 9]
print(sum(d * d for d in digits), sum(digits) ** 2)      # 82 và 100: khác nhau

82 100


### 3.4. Phép so sánh `>=` và các lỗi biên

`a >= 50` đúng khi `a == 50`. Các bài "ít nhất", "không nhỏ hơn" luôn là `>=`; "lớn hơn hẳn" mới là `>`. Hãy tự tìm một số cho hiệu **đúng bằng 50** để thử biên.

In [5]:
print(50 >= 50, 50 > 50, 49 >= 50)

True False False


### 3.5. Bảng đóng góp (lookup table)

Mỗi chữ số `d` có đóng góp cố định `d*d - d` vào hiệu. Dựng sẵn `TABLE = [d*d - d for d in range(10)]` rồi `TABLE[d]` truy cập `O(1)`.

In [6]:
TABLE = [d * d - d for d in range(10)]
print(TABLE)

[0, 0, 2, 6, 12, 20, 30, 42, 56, 72]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Làm đúng theo đề: tách chữ số, tính `digitSum`, tính `squareSum`, trừ rồi so với `50`.

Tính tay `n = 19`: chữ số `1, 9`; `digitSum = 10`; `squareSum = 1 + 81 = 82`; `82 - 10 = 72 >= 50` -> tốt.
Tính tay `n = 56`: `digitSum = 11`; `squareSum = 25 + 36 = 61`; `61 - 11 = 50` -> đúng bằng `50`, **tốt**.
Tính tay `n = 17`: `digitSum = 8`; `squareSum = 1 + 49 = 50`; `50 - 8 = 42 < 50` -> không tốt.

| `n` | chữ số | `digitSum` | `squareSum` | hiệu | tốt? |
|---|---|---|---|---|---|
| 19 | 1, 9 | 10 | 82 | 72 | có |
| 56 | 5, 6 | 11 | 61 | 50 | có (biên) |
| 17 | 1, 7 | 8 | 50 | 42 | không |
| 1000 | 1, 0, 0, 0 | 1 | 1 | 0 | không |

In [7]:
def diff(n):
    ds = [int(c) for c in str(n)]
    return sum(d * d for d in ds) - sum(ds)

for n in (19, 56, 17, 1000):
    print(n, diff(n), diff(n) >= 50)

19 72 True
56 50 True
17 42 False
1000 0 False


## 5. Cách 1 — Brute Force / baseline: hai tổng riêng

1. Tách các chữ số vào list `digits`.
2. `digitSum = sum(digits)`, `squareSum = sum(d * d for d in digits)`.
3. Trả `squareSum - digitSum >= 50`.

Đúng và dễ đọc nhất; "nút thắt" duy nhất là duyệt list hai lần và lưu list chữ số (hoàn toàn không đáng kể với tối đa 10 chữ số).

In [8]:
class SolutionTwoSums:
    def checkGoodInteger(self, n):
        digits = [int(c) for c in str(n)]
        digit_sum = sum(digits)
        square_sum = sum(d * d for d in digits)
        return square_sum - digit_sum >= 50

print(SolutionTwoSums().checkGoodInteger(1000), SolutionTwoSums().checkGoodInteger(19))

False True


**Độ phức tạp:** Time `O(L)` với `L` là số chữ số (`L <= 10`), Space `O(L)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** vì cả hai tổng cùng cộng trên **từng chữ số**, ta gộp:

`squareSum - digitSum = sum(d*d) - sum(d) = sum(d*d - d) = sum(d * (d - 1))`.

Mỗi chữ số `d` góp một lượng **độc lập** `f(d) = d*(d-1)`. Bảng `f` cho `d = 0..9` là:

| `d` | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|---|
| `f(d)` | 0 | 0 | 2 | 6 | 12 | 20 | 30 | 42 | 56 | 72 |

Hệ quả:

- chỉ cần **một lượt duyệt** cộng `f(d)`, không lưu hai tổng;
- `f(d) >= 0` nên tổng chỉ **tăng**: ngay khi tổng đạt `>= 50` có thể **trả `True` sớm**;
- chữ số `0` và `1` không góp gì.

**Invariant:** `total` luôn bằng tổng `f` trên các chữ số đã xử lý; vì `f >= 0` nên `total` không giảm.

## 7. Các cách tiếp theo

### Cách 2 — Một lượt duyệt bằng `divmod`

Duyệt từng chữ số, cộng dồn **hiệu** `d*d - d` vào một biến.

In [9]:
class SolutionOnePass:
    def checkGoodInteger(self, n):
        total = 0
        while n:
            n, d = divmod(n, 10)
            total += d * d - d
        return total >= 50

print(SolutionOnePass().checkGoodInteger(1000), SolutionOnePass().checkGoodInteger(19))

False True


**Độ phức tạp:** Time `O(L)`, Space `O(1)`.

### Cách 3 — Bảng đóng góp `TABLE[d]`

Dựng bảng 10 ô một lần, tra cứu cho từng chữ số. Dùng chuỗi `str(n)` để duyệt.

In [10]:
CONTRIB = [d * d - d for d in range(10)]

class SolutionTable:
    def checkGoodInteger(self, n):
        return sum(CONTRIB[int(c)] for c in str(n)) >= 50

print(SolutionTable().checkGoodInteger(1000), SolutionTable().checkGoodInteger(19))

False True


**Độ phức tạp:** Time `O(L)`, Space `O(1)` (bảng 10 ô; chuỗi `O(L)`).

### Cách 4 — Dừng sớm (early exit)

Vì mọi đóng góp `>= 0`, tổng chỉ tăng: ngay khi `total >= 50` trả `True`, không cần xét các chữ số còn lại.

In [11]:
class SolutionEarly:
    def checkGoodInteger(self, n):
        total = 0
        while n:
            n, d = divmod(n, 10)
            total += d * (d - 1)
            if total >= 50:
                return True
        return False

print(SolutionEarly().checkGoodInteger(1000), SolutionEarly().checkGoodInteger(19))

False True


**Độ phức tạp:** Time `O(L)` (best case `O(1)`), Space `O(1)`. Với tối đa 10 chữ số, lợi ích dừng sớm chỉ mang tính minh họa, nhưng mẫu "tổng đơn điệu thì dừng khi đạt ngưỡng" rất hay gặp.

## 8. TRACE từng bước

Trace Cách 4 với `n = 56` (hiệu đúng bằng ngưỡng):

| lần lặp | `n` trước | chữ số `d` | `d*(d-1)` | `total` | đạt `>= 50`? |
|---|---|---|---|---|---|
| 1 | 56 | 6 | 30 | 30 | chưa |
| 2 | 5 | 5 | 20 | 50 | **đạt -> True** |

Với `n = 1000`: chữ số `0,0,0,1` đều góp `0` -> `total = 0` -> `False`.

In [12]:
def trace_early(n):
    print("n =", n)
    total, step = 0, 0
    while n:
        step += 1
        n, d = divmod(n, 10)
        total += d * (d - 1)
        print(f"  lần {step}: chữ số={d}, d*(d-1)={d * (d - 1)}, total={total}")
        if total >= 50:
            print("  -> đạt ngưỡng, return True")
            return True
    print("  -> hết chữ số, return False")
    return False

assert trace_early(56) is True
assert trace_early(1000) is False
assert trace_early(17) is False

n = 56
  lần 1: chữ số=6, d*(d-1)=30, total=30
  lần 2: chữ số=5, d*(d-1)=20, total=50
  -> đạt ngưỡng, return True
n = 1000
  lần 1: chữ số=0, d*(d-1)=0, total=0
  lần 2: chữ số=0, d*(d-1)=0, total=0
  lần 3: chữ số=0, d*(d-1)=0, total=0
  lần 4: chữ số=1, d*(d-1)=0, total=0
  -> hết chữ số, return False
n = 17
  lần 1: chữ số=7, d*(d-1)=42, total=42
  lần 2: chữ số=1, d*(d-1)=0, total=42
  -> hết chữ số, return False


In [13]:
def trace_two_sums(n):
    digits = [int(c) for c in str(n)]
    ds, ss = sum(digits), sum(d * d for d in digits)
    print(f"n={n}: digits={digits}")
    print(f"  digitSum  = {' + '.join(map(str, digits))} = {ds}")
    print(f"  squareSum = {' + '.join(str(d * d) for d in digits)} = {ss}")
    print(f"  hiệu = {ss} - {ds} = {ss - ds}  ->  {ss - ds >= 50}")
    return ss - ds >= 50

assert trace_two_sums(19) is True

n=19: digits=[1, 9]
  digitSum  = 1 + 9 = 10
  squareSum = 1 + 81 = 82
  hiệu = 82 - 10 = 72  ->  True


## 9. Code nộp LeetCode

Dùng Cách 2 (một lượt duyệt, `O(1)` bộ nhớ).

In [14]:
from typing import List, Optional


class Solution:
    def checkGoodInteger(self, n: int) -> bool:
        total = 0
        while n:
            n, d = divmod(n, 10)
            total += d * d - d
        return total >= 50


print(Solution().checkGoodInteger(1000), Solution().checkGoodInteger(19))

False True


## 10. Vì sao thuật toán đúng

Gọi các chữ số của `n` là `d_1, ..., d_L`.

1. **Tách chữ số đúng.** `divmod(n, 10)` cho chữ số cuối `d` và phần còn lại; lặp tới `n = 0` đi qua mọi chữ số đúng một lần.
2. **Đại số.** `squareSum - digitSum = sum(d_i^2) - sum(d_i) = sum(d_i^2 - d_i)` (tách tổng thành tổng các hiệu theo từng chỉ số `i`). Mỗi bước cộng `d*d - d` nên `total` sau khi xử lý hết bằng đại lượng đó.
3. **So sánh.** `total >= 50` chính là điều kiện "tốt" của đề (bao gồm cả `total == 50`).
4. **Dừng sớm hợp lệ (Cách 4).** Vì `d*(d-1) >= 0` với mọi chữ số nguyên `d >= 0`, tổng riêng phần không giảm. Nếu đã `>= 50` thì tổng cuối cũng `>= 50`, nên trả `True` sớm là chính xác.
5. **Các cách khác** (hai tổng riêng, bảng tra) tính cùng một biểu thức theo cách khác.

## 11. Phân tích Time / Space Complexity

Gọi `L = số chữ số của n <= 10` (do `n <= 10^9`).

- Cách 1: Time `O(L)`, Space `O(L)` (list chữ số).
- Cách 2 (nộp): Time `O(L)`, Space `O(1)`.
- Cách 3: Time `O(L)`, Space `O(1)` (bảng hằng số; chuỗi `O(L)`).
- Cách 4: Time `O(L)`, best case `O(1)`, Space `O(1)`.

Tất cả đều `O(log n)` theo giá trị `n`. Với tối đa 10 chữ số, chênh lệch thực tế không đáng kể; quan trọng là đúng và gọn.

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

In [15]:
import random

solutions = [SolutionTwoSums(), SolutionOnePass(), SolutionTable(), SolutionEarly(), Solution()]

fixed = [
    (1000, False), (19, True), (56, True), (17, False), (1, False), (9, True),   # 9: 81 - 9 = 72
    (7, False), (8, True), (65, True), (55, False), (11, False), (99, True),
    (555, True), (111111111, False), (10 ** 9, False), (999999999, True),
    (44444, True), (4444, False), (33333333, False), (3333333, False), (333333333, True),
]
for n, expected in fixed:
    for sol in solutions:
        assert sol.checkGoodInteger(n) == expected, (type(sol).__name__, n)
print("test cố định OK:", len(fixed), "ca")

random.seed(3959)
cnt = {True: 0, False: 0}
for _ in range(1500):
    n = random.choice([random.randint(1, 10 ** 3), random.randint(1, 10 ** 6), random.randint(1, 10 ** 9)])
    expected = diff(n) >= 50                       # trọng tài: theo đúng định nghĩa
    cnt[expected] += 1
    for sol in solutions:
        assert sol.checkGoodInteger(n) == expected, (type(sol).__name__, n)
print("test ngẫu nhiên OK, phân bố:", cnt)
assert cnt[True] > 100 and cnt[False] > 100

test cố định OK: 21 ca
test ngẫu nhiên OK, phân bố: {True: 1305, False: 195}


### Test thuộc tính (property test)

(1) Hoán vị chữ số không đổi kết quả (tổng không phụ thuộc thứ tự). (2) Thêm chữ số `0` hoặc `1` (đóng góp bằng 0) vào bất kỳ chỗ nào không đổi kết quả. (3) Thêm một chữ số **không làm giảm** hiệu, nên nếu `n` tốt thì mọi số chứa chữ số của `n` (và thêm chữ số khác) cũng tốt.

In [16]:
random.seed(39590)
for _ in range(500):
    n = random.randint(1, 10 ** 8)
    r = Solution().checkGoodInteger(n)
    digits = list(str(n)); random.shuffle(digits)
    if digits[0] != "0":
        assert Solution().checkGoodInteger(int("".join(digits))) == r
    assert Solution().checkGoodInteger(n * 10) == r                 # thêm chữ số 0 ở cuối
    assert Solution().checkGoodInteger(int("1" + str(n))) == r      # thêm chữ số 1 ở đầu
    if r:
        assert Solution().checkGoodInteger(n * 10 + random.randint(0, 9)) is True
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `>` thay `>=`

Hiệu đúng bằng `50` vẫn là số tốt. `n = 56` cho hiệu `50`.

In [17]:
def wrong_strict(n):
    ds = [int(c) for c in str(n)]
    return sum(d * d for d in ds) - sum(ds) > 50             # SAI: phải >= 50

print(wrong_strict(56), "!=", Solution().checkGoodInteger(56))
assert wrong_strict(56) is False and Solution().checkGoodInteger(56) is True
assert diff(56) == 50

False != True


### Lỗi 2: nhầm tổng bình phương với **bình phương của tổng**

`(1 + 7) ** 2 = 64` còn `1 + 49 = 50`. Với `n = 17`: dùng `(tổng) ** 2` cho hiệu `64 - 8 = 56 >= 50` (sai), trong khi đúng là `50 - 8 = 42`.

In [18]:
def wrong_square_of_sum(n):
    ds = [int(c) for c in str(n)]
    return sum(ds) ** 2 - sum(ds) >= 50                      # SAI: (tổng)^2 thay vì tổng các bình phương

print(wrong_square_of_sum(17), "!=", Solution().checkGoodInteger(17))
assert wrong_square_of_sum(17) is True and Solution().checkGoodInteger(17) is False

True != False


### Lỗi 3: đảo thứ tự phép trừ

Luôn `squareSum - digitSum` (không âm vì `d*d >= d` với chữ số nguyên). Đảo ngược cho số âm, không bao giờ `>= 50`.

In [19]:
def wrong_order(n):
    ds = [int(c) for c in str(n)]
    return sum(ds) - sum(d * d for d in ds) >= 50            # SAI: ngược chiều

print(wrong_order(19), "!=", Solution().checkGoodInteger(19))
assert wrong_order(19) is False and Solution().checkGoodInteger(19) is True

False != True


### Lỗi 4: lấy số chữ số thay vì giá trị chữ số

`len(str(n))` cho độ dài, không phải tổng. Với `n = 19`: `len = 2`, hoàn toàn không liên quan tới `72`.

In [20]:
def wrong_len(n):
    L = len(str(n))
    return L * L - L >= 50                                    # SAI: dùng độ dài chứ không phải chữ số

print(wrong_len(19), "!=", Solution().checkGoodInteger(19))
assert wrong_len(19) is False and Solution().checkGoodInteger(19) is True

False != True


### Lỗi 5: dùng `while n:` nhưng quên cập nhật `n`

Nếu viết `d = n % 10` mà không có `n //= 10` thì vòng lặp **không bao giờ dừng**. Dùng `divmod` giúp cả hai việc làm cùng lúc. (Ta chỉ mô phỏng bằng giới hạn số vòng để notebook không bị treo.)

In [21]:
def wrong_no_update(n, limit=5):
    total, steps = 0, 0
    while n and steps < limit:                                # giới hạn để minh họa vòng lặp vô hạn
        d = n % 10                                            # SAI: quên n //= 10
        total += d * d - d
        steps += 1
    return total, steps

print(wrong_no_update(19))
assert wrong_no_update(19) == (360, 5) and Solution().checkGoodInteger(19) is True

(360, 5)


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng của A trừ tổng của B trên cùng tập phần tử" -> gộp thành **một tổng** `sum(A_i - B_i)`.
- Ràng buộc nhỏ (10 chữ số): đúng quan trọng hơn nhanh; nhưng mẫu **dừng sớm khi tổng đơn điệu đạt ngưỡng** là thói quen tốt.
- Chữ số chỉ có 10 giá trị nên có thể dựng **bảng tra** `[f(d) for d in range(10)]`.
- Tự tìm **ca biên** của ngưỡng: ở đây `n = 56` cho hiệu đúng `50`; hãy thử `>=` và `>`.
- `(a + b) ** 2` khác `a ** 2 + b ** 2`: đọc kỹ "tổng bình phương" hay "bình phương của tổng".
- Luôn thử `n` một chữ số (`1`, `9`) và `n = 10^9`.

## 15. Biến thể của bài

1. **Happy Number** (202): lặp `n -> tổng bình phương các chữ số`.
2. **Add Digits** (258), **Digit Frequency Score** (3945, cùng đợt): tổng chữ số.
3. **Armstrong Number** (1134, Premium): so tổng lũy thừa các chữ số với `n`.
4. **Harshad Number** (3099): `n` chia hết cho tổng chữ số.
5. Biến thể: ngưỡng khác (`squareSum - digitSum >= T`), hoặc **đếm** các số từ `1` đến `N` là "tốt" (vét cạn với `N` nhỏ).
6. Biến thể: dùng tích các chữ số thay vì tổng (ca `0` làm tích về `0`).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai tổng riêng | `digitSum`, `squareSum` rồi trừ | `O(L)` | `O(L)` | Đọc đúng đề, dễ gỡ lỗi |
| 2. Một lượt `divmod` | cộng `d*d - d` | `O(L)` | `O(1)` | **Nộp bài** |
| 3. Bảng đóng góp | `TABLE[d]` | `O(L)` | `O(1)` | Khi hàm đóng góp phức tạp |
| 4. Dừng sớm | thoát khi `total >= 50` | `O(L)` (best `O(1)`) | `O(1)` | Khi tổng đơn điệu và ngưỡng nhỏ |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().checkGoodInteger(88)` và `Solution().checkGoodInteger(2345)` ra gì?

**Bài 2 (tính tay):** tìm một số hai chữ số khác `56` mà hiệu **đúng bằng 50** (gợi ý: dùng bảng `f`).

**Bài 3 (điền TODO):** hoàn thành `minimal_good_two_digit()` trả về **số hai chữ số nhỏ nhất** là số tốt.

**Bài 4 (tự viết lại):** viết `how_good(n)` trả về hiệu `squareSum - digitSum` (không so sánh), rồi `is_good(n, T)` với ngưỡng `T` tùy ý.

**Bài 5 (mini challenge):** đếm có bao nhiêu số trong `[1, 1000]` là số tốt, và số tốt nhỏ nhất có 3 chữ số.

In [22]:
# Bài 1: đoán rồi chạy
print(Solution().checkGoodInteger(88), Solution().checkGoodInteger(2345))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def minimal_good_two_digit_todo():
    # TODO: duyệt n từ 10 đến 99, trả về n đầu tiên thỏa checkGoodInteger
    return -1

True False


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: dùng bảng `f`: `f(8) = 56`, `f(5) = 20`. Bài 2: hai chữ số `a, b` cần `f(a) + f(b) = 50`.

**Mức 2 (vừa):** Bài 3: duyệt `for n in range(10, 100)` và trả về số đầu tiên thỏa. Bài 4: `how_good` dùng `d * d - d`; `is_good` so `>= T`. Bài 5: `sum(1 for n in range(1, 1001) if ...)`.

**Mức 3 (gần đáp án):** Bài 1: `88 -> True` (`56 + 56 = 112`), `2345 -> 2 + 6 + 12 + 20 = 40 -> False`. Bài 2: `f(6) + f(5) = 50` nên `56` và `65` đều được. Bài 3: ... thử `n = 38`? (`f(3) + f(8) = 6 + 56 = 62`); số nhỏ hơn có thể có, hãy chạy code.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().checkGoodInteger(88) is True and Solution().checkGoodInteger(2345) is False
assert diff(2345) == 40 and diff(65) == 50 and diff(56) == 50

# Bài 3
def minimal_good_two_digit():
    for n in range(10, 100):
        if Solution().checkGoodInteger(n):
            return n
    return -1

first = minimal_good_two_digit()
assert first == min(n for n in range(10, 100) if diff(n) >= 50)
print("số hai chữ số tốt nhỏ nhất:", first)

# Bài 4
def how_good(n):
    return sum(int(c) * int(c) - int(c) for c in str(n))

def is_good(n, T):
    return how_good(n) >= T

assert how_good(19) == 72 and how_good(1000) == 0 and is_good(56, 50) and not is_good(56, 51)

# Bài 5
good_count = sum(1 for n in range(1, 1001) if Solution().checkGoodInteger(n))
first_three = next(n for n in range(100, 1000) if Solution().checkGoodInteger(n))
assert good_count == sum(1 for n in range(1, 1001) if how_good(n) >= 50)
print("đếm số tốt trong [1, 1000]:", good_count, "| số tốt 3 chữ số nhỏ nhất:", first_three)
print("đáp án bài tập OK")

số hai chữ số tốt nhỏ nhất: 18
đếm số tốt trong [1, 1000]: 677 | số tốt 3 chữ số nhỏ nhất: 108
đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def checkGoodInteger(self, n: int) -> bool:
        total = 0
        while n:
            n, d = divmod(n, 10)    # tách chữ số cuối
            total += d * d - d      # squareSum - digitSum, gộp theo từng chữ số
        return total >= 50          # bao gồm cả ca đúng bằng 50
```

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] So sánh là `>= 50` (đã thử `n = 56`), dùng tổng **bình phương** chứ không phải bình phương của tổng.
- [ ] `n` một chữ số (`1`, `9`) và `n = 10**9` chạy đúng.